# 📷 CIFAR-100 — Détection temps réel
**ResNet18 · 100 classes · Caméra mobile**

Ce notebook fonctionne en deux modes :
- **Mode Colab** : lance Gradio avec un lien public accessible sur téléphone
- **Mode local** : lance Gradio en local depuis PyCharm


## 0 · Environnement

In [ ]:
import sys, os

# Détection automatique de l'environnement
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environnement :', 'Google Colab' if IN_COLAB else 'Local (PyCharm)')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device        :', DEVICE)
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
else:
    print('⚠️  Pas de GPU — CPU utilisé (plus lent)')


## 1 · Installation des dépendances

In [ ]:
# En local : pip install gradio torch torchvision pillow numpy
# En Colab  : exécuté automatiquement ci-dessous

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

required = ['gradio', 'torch', 'torchvision', 'pillow', 'numpy']
for pkg in required:
    try:
        __import__(pkg.split('[')[0])
        print(f'✅ {pkg} déjà installé')
    except ImportError:
        print(f'📦 Installation de {pkg}...')
        pip_install(pkg)
        print(f'✅ {pkg} installé')


## 2 · Chemin vers ton modèle

**Modifie uniquement cette cellule.**

- **Colab** : monte Drive puis donne le chemin Drive
- **Local** : donne le chemin absolu sur ton PC


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  MODIFIE ICI — chemin vers ton fichier best.pth ou best.keras
# ═══════════════════════════════════════════════════════════════

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Chemin Drive — modifie selon où tu as sauvegardé best.pth
    MODEL_PATH = '/content/drive/MyDrive/cifar100/checkpoints/best.pth'

else:
    # Chemin local Windows — modifie avec ton vrai chemin
    MODEL_PATH = r'C:\Users\ZIDAN\PycharmProjects\Computer_Vision_project\cifar100-project\checkpoints\best.pth'
    # Ou Linux/Mac :
    # MODEL_PATH = '/home/user/cifar100-project/checkpoints/best.pth'

# ─── Vérification ─────────────────────────────────────────────────────────────
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    ext     = os.path.splitext(MODEL_PATH)[1].lower()
    print(f'✅ Modèle trouvé : {MODEL_PATH}')
    print(f'   Taille  : {size_mb:.1f} MB')
    print(f'   Format  : {ext}')
    MODEL_FORMAT = ext  # '.pth' ou '.keras' ou '.h5'
else:
    print(f'❌ Fichier non trouvé : {MODEL_PATH}')
    print('   Vérifie le chemin ci-dessus')
    MODEL_FORMAT = None


## 3 · 100 classes CIFAR-100

In [ ]:
CIFAR100_CLASSES = [
    'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle',
    'bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel',
    'can','castle','caterpillar','cattle','chair','chimpanzee','clock',
    'cloud','cockroach','couch','crab','crocodile','cup','dinosaur',
    'dolphin','elephant','flatfish','forest','fox','girl','hamster',
    'house','kangaroo','keyboard','lamp','lawn_mower','leopard','lion',
    'lizard','lobster','man','maple_tree','motorcycle','mountain',
    'mouse','mushroom','oak_tree','orange','orchid','otter','palm_tree',
    'pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine',
    'possum','rabbit','raccoon','ray','road','rocket','rose','sea','seal',
    'shark','shrew','skunk','skyscraper','snail','snake','spider',
    'squirrel','streetcar','sunflower','sweet_pepper','table','tank',
    'telephone','television','tiger','tractor','train','trout','tulip',
    'turtle','wardrobe','whale','willow_tree','wolf','woman','worm',
]
print(f'{len(CIFAR100_CLASSES)} classes chargées ✅')


## 4 · Chargement automatique du modèle (PyTorch ou Keras)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from PIL import Image

# ─── Architecture ResNet18 ─────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))

class ResNet18(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.in_ch  = 64
        self.conv1  = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(64)
        self.layer1 = self._make(64,  2, 1)
        self.layer2 = self._make(128, 2, 2)
        self.layer3 = self._make(256, 2, 2)
        self.layer4 = self._make(512, 2, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(512, num_classes)
    def _make(self, ch, n, stride):
        layers = [ResidualBlock(self.in_ch, ch, stride)]
        self.in_ch = ch
        for _ in range(1, n): layers.append(ResidualBlock(ch, ch))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        for l in [self.layer1, self.layer2, self.layer3, self.layer4]: x = l(x)
        return self.fc(torch.flatten(self.pool(x), 1))

# ─── Chargement selon le format détecté ───────────────────────────────────────
model     = None
framework = None

if MODEL_FORMAT in ['.pth', '.pt']:
    # ── PyTorch ──────────────────────────────────────────────────────────────
    framework = 'pytorch'
    model = ResNet18(num_classes=100).to(DEVICE)
    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)

    # Gestion des différents formats de checkpoint
    if isinstance(ckpt, dict):
        state = ckpt.get('model', ckpt.get('state_dict', ckpt))
    else:
        state = ckpt

    # Nettoyage des préfixes 'module.' (DataParallel)
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    print('✅ ResNet18 PyTorch chargé')

elif MODEL_FORMAT in ['.keras', '.h5']:
    # ── Keras / TensorFlow ────────────────────────────────────────────────────
    framework = 'keras'
    import tensorflow as tf
    model = tf.keras.models.load_model(MODEL_PATH)
    print('✅ Modèle Keras chargé')
    print('   Input shape  :', model.input_shape)
    print('   Output shape :', model.output_shape)

else:
    print('❌ Format non reconnu — vérifie MODEL_FORMAT =', MODEL_FORMAT)
    print('   Formats supportés : .pth, .pt, .keras, .h5')


## 5 · Fonction d'inférence universelle

In [ ]:
import torchvision.transforms as T

MEAN = (0.5071, 0.4867, 0.4408)
STD  = (0.2675, 0.2565, 0.2761)

tf_pytorch = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def predict(pil_img, top_k=5):
    """
    Prédit les top_k classes pour une image PIL.
    Compatible PyTorch (.pth) et Keras (.keras/.h5).
    Retourne une liste de (label, score_float).
    """
    img_rgb = pil_img.convert('RGB')

    if framework == 'pytorch':
        x      = tf_pytorch(img_rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()

    elif framework == 'keras':
        input_h, input_w = model.input_shape[1], model.input_shape[2]
        arr   = np.array(img_rgb.resize((input_w, input_h))).astype('float32') / 255.0
        probs = model.predict(np.expand_dims(arr, 0), verbose=0)[0]

    top_idx = probs.argsort()[::-1][:top_k]
    return [(CIFAR100_CLASSES[i], float(probs[i])) for i in top_idx]

# ─── Test rapide ───────────────────────────────────────────────────────────────
test = Image.new('RGB', (224, 224), (100, 150, 80))
results = predict(test, top_k=3)
print('Test inférence OK :', results)


## 6 · Application Gradio — détection en temps réel 📱

In [ ]:
import gradio as gr
from PIL import Image, ImageDraw

# ─── Palette de couleurs par confiance ────────────────────────────────────────
def conf_color(score):
    if score > 0.6:  return (34,  197, 94)   # vert  — très confiant
    if score > 0.3:  return (234, 179, 8)    # jaune — moyen
    return                  (239, 68,  68)   # rouge — peu confiant

def draw_results_overlay(pil_img, results):
    """Dessine les résultats directement sur l'image caméra."""
    img    = pil_img.copy().resize((480, 360))
    draw   = ImageDraw.Draw(img)
    W, H   = img.size
    pad    = 10

    # Fond semi-transparent en bas
    overlay_h = min(len(results) * 38 + 20, 220)
    draw.rectangle([0, H - overlay_h, W, H], fill=(0, 0, 0, 180))

    for i, (label, score) in enumerate(results):
        y      = H - overlay_h + 10 + i * 38
        color  = conf_color(score)
        bar_w  = int(score * (W - 2 * pad - 120))

        # Barre de confiance
        draw.rectangle([pad + 110, y + 6, pad + 110 + bar_w, y + 28], fill=color)
        draw.rectangle([pad + 110, y + 6, W - pad, y + 28],
                       outline=(80, 80, 80), width=1)

        # Texte label + pourcentage
        draw.text((pad, y + 4),  f'{i+1}. {label[:18]}', fill=(255, 255, 255))
        draw.text((W - 58, y + 4), f'{score*100:5.1f}%',  fill=color)

    return img

def predict_stream(frame):
    """Appelée par Gradio à chaque frame caméra."""
    if frame is None:
        return None, '⏳ En attente de la caméra...'

    pil_img = Image.fromarray(frame) if not isinstance(frame, Image.Image) else frame

    try:
        results = predict(pil_img, top_k=5)
    except Exception as e:
        return pil_img, f'❌ Erreur : {e}'

    # Image avec overlay
    annotated = draw_results_overlay(pil_img, results)

    # Texte résumé
    top        = results[0]
    confidence = '🟢' if top[1] > 0.6 else ('🟡' if top[1] > 0.3 else '🔴')
    summary    = (
        f'{confidence} {top[0].upper()}  —  {top[1]*100:.1f}%\n\n'
        + '\n'.join(f'  {i+1}. {l:<20} {s*100:.1f}%'
                     for i, (l, s) in enumerate(results))
    )

    return annotated, summary

# ─── Interface ────────────────────────────────────────────────────────────────
css = """
body, .gradio-container {
    background: #0a0a0f !important;
    font-family: 'Courier New', monospace !important;
}
h1 {
    text-align: center;
    background: linear-gradient(90deg, #38bdf8, #34d399);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 1.6rem;
    letter-spacing: 3px;
    padding: 12px 0;
}
.subtitle {
    text-align: center;
    color: #475569;
    font-size: 0.82rem;
    letter-spacing: 1px;
    margin-bottom: 16px;
}
.gradio-image img { border-radius: 12px !important; }
.gradio-textbox textarea {
    background: #0f172a !important;
    color: #94a3b8 !important;
    font-family: 'Courier New', monospace !important;
    font-size: 0.9rem !important;
    border: 1px solid #1e293b !important;
    border-radius: 8px !important;
}
.gradio-button {
    background: #1e293b !important;
    color: #38bdf8 !important;
    border: 1px solid #38bdf8 !important;
    border-radius: 6px !important;
}
footer { display: none !important; }
"""

with gr.Blocks(title='CIFAR-100 Detector', css=css) as demo:

    gr.HTML('<h1>◈ CIFAR-100 DETECTOR</h1>')
    gr.HTML('<div class="subtitle">ResNet18 · 100 classes · temps réel</div>')

    with gr.Row(equal_height=True):

        with gr.Column(scale=3):
            cam = gr.Image(
                sources=['webcam'],
                streaming=True,
                label='',
                mirror_webcam=False,
                height=380,
                show_label=False,
            )

        with gr.Column(scale=2):
            out_img  = gr.Image(
                label='Détection',
                height=240,
                show_label=False,
            )
            out_text = gr.Textbox(
                label='',
                lines=8,
                interactive=False,
                show_label=False,
                placeholder='En attente...',
            )

    with gr.Row():
        gr.HTML(
            '<p style="text-align:center;color:#334155;font-size:0.75rem;">'
            '📱 Sur téléphone : autoriser la caméra dans le navigateur  ·  '
            '🟢 > 60%  🟡 30-60%  🔴 < 30%'
            '</p>'
        )

    # Streaming temps réel
    cam.stream(
        fn=predict_stream,
        inputs=[cam],
        outputs=[out_img, out_text],
        stream_every=0.4,
    )

# ─── Lancement ────────────────────────────────────────────────────────────────
print()
print('=' * 55)

if IN_COLAB:
    print('🚀 Lancement Colab — lien public généré ci-dessous')
    print('📱 Ouvre le lien sur ton téléphone')
    print('=' * 55)
    demo.launch(
        share=True,
        debug=False,
        show_error=True,
    )
else:
    print('🚀 Lancement local — ouvre http://localhost:7860')
    print('📱 Sur téléphone (même réseau Wi-Fi) :')
    import socket
    try:
        ip = socket.gethostbyname(socket.gethostname())
        print(f'   http://{ip}:7860')
    except:
        print('   http://<IP_DE_TON_PC>:7860')
    print('=' * 55)
    demo.launch(
        server_name='0.0.0.0',  # écoute sur toutes les interfaces réseau
        server_port=7860,
        share=False,
        debug=False,
    )


## 7 · (Optionnel) Test avec image uploadée — sans caméra

In [ ]:
# Lance cette cellule séparément pour tester avec une image

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    sources  = {n: __import__('io').BytesIO(d) for n, d in uploaded.items()}
else:
    # Mets le chemin d'une image locale
    sources = {'test.jpg': r'C:\Users\ZIDAN\Pictures\test.jpg'}

import matplotlib.pyplot as plt

for name, src in sources.items():
    img     = Image.open(src).convert('RGB')
    results = predict(img, top_k=5)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.imshow(img); ax1.set_title(name); ax1.axis('off')

    labels = [r[0] for r in results[::-1]]
    scores = [r[1] for r in results[::-1]]
    colors = [('green' if s > 0.6 else ('orange' if s > 0.3 else 'red')) for s in scores]
    ax2.barh(labels, scores, color=colors)
    ax2.set_xlim(0, 1); ax2.set_title('Top-5 ResNet18')
    for j, (l, s) in enumerate(zip(labels, scores)):
        ax2.text(s + 0.01, j, f'{s*100:.1f}%', va='center', fontsize=9)

    plt.tight_layout(); plt.show()
    print(f'\nTop-1 → {results[0][0]} ({results[0][1]*100:.1f}%)')
